# Build candidates (Organization level)

In [1]:
from pathlib import Path
import sys
from pyspark.sql import functions as sf


In [2]:
# Make repo root importable regardless of Jupyter working directory
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebooks.spark_session import create_spark_session

spark = create_spark_session('build-candidates-org')
spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/jupyter/ivy/cache
The jars for the packages stored in: /tmp/jupyter/ivy/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dceaa623-86d7-4c26-b607-3011725a9bd2;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (116ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (16725ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Fina

In [3]:
print('master:', spark.sparkContext.master)
print('app:', spark.sparkContext.appName)

master: spark://spark-master:7077
app: build-candidates-org


In [4]:
path = "s3a://gba-bronze-zone-prod/gh_events_flat/dt=2026-03-17/hr=22/"
df = spark.read.parquet(path)
df.select("*").distinct().show(5, truncate=False)

26/03/21 23:32:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

+----------+----------+--------------------+----------+----------------------------------------------+---------+-------------------+---------+---------+-------------------------------------+--------------------------------------------------+---------+--------------+----------------+---------------+-------------------+--------+----------+----------+---------+--------------------------+---------------------------------------------------------------------------+
|event_id  |event_type|created_at          |repo_id   |repo_full_name                                |actor_id |actor_login        |org_id   |org_login|org_url                              |org_avatar_url                                    |is_public|payload_action|payload_ref_type|pull_request_id|pull_request_merged|issue_id|comment_id|release_id|member_id|ingestion_ts              |source_file                                                                |
+----------+----------+--------------------+----------+-----------------

In [13]:
df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- org_url: string (nullable = true)
 |-- org_avatar_url: string (nullable = true)
 |-- is_public: boolean (nullable = true)
 |-- payload_action: string (nullable = true)
 |-- payload_ref_type: string (nullable = true)
 |-- pull_request_id: long (nullable = true)
 |-- pull_request_merged: boolean (nullable = true)
 |-- issue_id: long (nullable = true)
 |-- comment_id: long (nullable = true)
 |-- release_id: long (nullable = true)
 |-- member_id: long (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [15]:
df.count()

151764

In [26]:
df.filter(
    sf.isnull(sf.col("repo_id"))
).count()

4

In [27]:
df.filter(
    sf.isnull(sf.col("repo_full_name"))
).count()

4

In [28]:
df.filter(
    sf.isnull(sf.col("org_id"))
).count()

118801

In [30]:
df.filter(
    sf.isnull(sf.col("org_login"))
).count()

118801

In [14]:
df.filter(sf.isnotnull(sf.col("repo_id"))).filter(sf.isnotnull(sf.col("repo_full_name"))).filter(sf.isnotnull(sf.col("org_id"))).filter(sf.isnotnull(sf.col("org_login"))).count()

32963

In [25]:
df.groupBy("org_id", "org_login").show()

AttributeError: 'GroupedData' object has no attribute 'show'

In [44]:
df.select("repo_id", "org_id", "org_login").show()

[Stage 10:>                                                         (0 + 1) / 1]

+----------+---------+--------------------+
|   repo_id|   org_id|           org_login|
+----------+---------+--------------------+
|1182068066|262480739|           avantkeel|
| 725205304|150920049|           unslothai|
|1130056523|     NULL|                NULL|
|1144277465|     NULL|                NULL|
|1181227484|     NULL|                NULL|
|1172849899|     NULL|                NULL|
| 858891245|     NULL|                NULL|
|1174512695|     NULL|                NULL|
| 914857305|     NULL|                NULL|
| 937425438|     NULL|                NULL|
| 963215921|     NULL|                NULL|
| 101792503| 30125649|          envoyproxy|
|1173759246|     NULL|                NULL|
|1147325301|     NULL|                NULL|
|1159470151|262001670|         supermarian|
|1183770633| 96167003|static-web-apps-t...|
|  45061433| 15253016|            cmu-phil|
|1137186531|     NULL|                NULL|
|1182001268|     NULL|                NULL|
| 414932421|     NULL|          

In [31]:
df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- repo_id: long (nullable = true)
 |-- repo_full_name: string (nullable = true)
 |-- actor_id: long (nullable = true)
 |-- actor_login: string (nullable = true)
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- org_url: string (nullable = true)
 |-- org_avatar_url: string (nullable = true)
 |-- is_public: boolean (nullable = true)
 |-- payload_action: string (nullable = true)
 |-- payload_ref_type: string (nullable = true)
 |-- pull_request_id: long (nullable = true)
 |-- pull_request_merged: boolean (nullable = true)
 |-- issue_id: long (nullable = true)
 |-- comment_id: long (nullable = true)
 |-- release_id: long (nullable = true)
 |-- member_id: long (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [15]:
df.count()

151764

In [5]:
import re

def camel_to_snake(value: str) -> str:
  return re.sub(r"(?<!^)(?=[A-Z])", "_", value).lower()

In [6]:
event_types_list = [item.event_type for item in df.select('event_type').distinct().collect()]
event_types_list

['PullRequestReviewEvent',
 'PushEvent',
 'GollumEvent',
 'ReleaseEvent',
 'CommitCommentEvent',
 'CreateEvent',
 'PullRequestReviewCommentEvent',
 'IssueCommentEvent',
 'DeleteEvent',
 'IssuesEvent',
 'ForkEvent',
 'PublicEvent',
 'MemberEvent',
 'WatchEvent',
 'PullRequestEvent',
 'DiscussionEvent']

In [7]:
dt = "2026-03-17"
hr = "22"
event_columns = []
ration_columns = {}
for raw_name in event_types_list:
    alias_name = f"{camel_to_snake(raw_name)}s"
    column = sf.sum(sf.when(sf.col("event_type") == raw_name, 1).otherwise(0)).alias(alias_name)
    ration_columns[f"{alias_name}_ratio"] = sf.when(
        sf.col("total_events") > 0, sf.col(alias_name) / sf.col("total_events")
    ).otherwise(sf.lit(0.0))
    event_columns.append(column)

In [8]:
org_metrics = df.groupBy("org_id", "org_login").agg(
    sf.count("*").alias("total_events"),
    sf.count("is_public").alias("public_events_count"),
    sf.countDistinct("repo_id").alias("total_repositories"),
    sf.countDistinct("actor_id").alias("unique_actors"),
    sf.countDistinct("issue_id").alias("issues_count"),
    sf.countDistinct("comment_id").alias("comments_count"),
    sf.countDistinct("release_id").alias("releases_count"),
    sf.countDistinct("pull_request_merged").alias("merged_pr_count"),
    sf.max("created_at").alias("last_event_at"),
    sf.sum(
        sf.when(
            sf.lower(sf.col("actor_login")).ilike("%[bot]"),
            1
        ).otherwise(0)
    ).alias("bot_events"),
    *event_columns
)

for key, col in ration_columns.items():    
    org_metrics = org_metrics.withColumn(key, col)

org_metrics = (
    org_metrics
        .withColumn(
            "composite_score",
              0.5 * sf.log1p(sf.col("total_events")) +
              0.3 * sf.col("push_events_ratio") +
              0.2 * sf.log1p(sf.col("unique_actors"))
        )
        .withColumn(
              "bot_ratio",
              sf.when(sf.col("total_events") > 0, sf.col("bot_events") / sf.col("total_events"))
               .otherwise(sf.lit(0.0))
        )
        .withColumn("dt", sf.lit(dt).cast("date"))
        .withColumn("hr", sf.lit(hr).cast("int"))
        .withColumn(
          "last_event_at",
          sf.to_timestamp("last_event_at", "yyyy-MM-dd'T'HH:mm:ssX")
        )
)

In [9]:
org_metrics.printSchema()

root
 |-- org_id: long (nullable = true)
 |-- org_login: string (nullable = true)
 |-- total_events: long (nullable = false)
 |-- public_events_count: long (nullable = false)
 |-- total_repositories: long (nullable = false)
 |-- unique_actors: long (nullable = false)
 |-- issues_count: long (nullable = false)
 |-- comments_count: long (nullable = false)
 |-- releases_count: long (nullable = false)
 |-- merged_pr_count: long (nullable = false)
 |-- last_event_at: timestamp (nullable = true)
 |-- bot_events: long (nullable = true)
 |-- pull_request_review_events: long (nullable = true)
 |-- push_events: long (nullable = true)
 |-- gollum_events: long (nullable = true)
 |-- release_events: long (nullable = true)
 |-- commit_comment_events: long (nullable = true)
 |-- create_events: long (nullable = true)
 |-- pull_request_review_comment_events: long (nullable = true)
 |-- issue_comment_events: long (nullable = true)
 |-- delete_events: long (nullable = true)
 |-- issues_events: long (null

In [10]:
org_metrics.select("org_id", "org_login", "dt", "hr", "public_events", "total_events", "public_events_count").show()

[Stage 7:=============================>                             (1 + 1) / 2]

+---------+-----------------+----------+---+-------------+------------+-------------------+
|   org_id|        org_login|        dt| hr|public_events|total_events|public_events_count|
+---------+-----------------+----------+---+-------------+------------+-------------------+
|150412084|        arki-tech|2026-03-17| 22|            0|           7|                  7|
|  8632328|            coveo|2026-03-17| 22|            0|           1|                  1|
|  6404994|         fini-net|2026-03-17| 22|            0|           6|                  6|
|108928776|          oven-sh|2026-03-17| 22|            0|          12|                 12|
|262892540| jsgods-rs-tandem|2026-03-17| 22|            0|           7|                  7|
|  1215332|      pyinstaller|2026-03-17| 22|            0|           3|                  3|
|186863079|    kaito-project|2026-03-17| 22|            0|           4|                  4|
|104528311|Astrofili-Centesi|2026-03-17| 22|            0|           8|         

In [26]:
 spark.stop()